In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from synonym import step1, step2, step3, step4, fix1, evaluation, prompts

load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_seed52_ratio0.3_synonymous_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10
fix_repetition = 3
current_df = df.copy()

for iteration in range(fix_repetition):
    print(f"\n{'='*40}")
    print(f">>> ITERATION {iteration + 1} / {fix_repetition}")
    print(f"{'='*40}")

    print("\n>>> STEP 1: Candidate Filtering")
    activity_list_json = json.dumps(current_df['activity'].unique().tolist(), indent=4, ensure_ascii=False)
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, prompts.SYSTEM_PROMPT_SYNONYMOUS_STEP1, prompts.USER_PROMPT_SYNONYMOUS_STEP1)
    
    if not res_s1['data']:
        print(f">>> No more synonymous candidates found in iteration {iteration + 1}. Stopping loop.")
        break
    print(f"Candidates found ({len(res_s1['data'])} total): {res_s1['data'][:5]} ...")

    print("\n>>> STEP 2: Context Abstraction")
    context_json = step2.get_synonym_context(current_df, set(res_s1['data']))
    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, context_json, prompts.SYSTEM_PROMPT_SYNONYM_STEP2, prompts.USER_PROMPT_SYNONYM_STEP2)
    s2_output_json = json.dumps(res_s2["summarized_context"], indent=2, ensure_ascii=False)
    print(f"Sample Context: {res_s2['summarized_context'][0] if res_s2['summarized_context'] else 'None'}")

    print("\n>>> STEP 3: Clustering")
    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, s2_output_json, prompts.SYSTEM_PROMPT_SYNONYM_STEP3, prompts.USER_PROMPT_SYNONYM_STEP3)
    
    if not res_s3:
        print(f">>> No clusters formed in iteration {iteration + 1}.")
        continue
    
    print(f"\n>>> Clusters Preview (Total: {len(res_s3)} groups)")
    for group in res_s3[:3]:
        items = group[:3] + ["..."] if len(group) > 3 else group[:3]
        print(f"  - {items}")
    print(f"  ... and {len(res_s3)-3} more groups")

    print("\n>>> STEP 4: Final Mapping")
    activity_counts = current_df['activity'].value_counts().to_dict()
    res_s4 = step4.run_step4(res_s3, activity_counts)
    
    print(f"\n>>> Mapping Preview (Total: {len(res_s4)} groups)")
    for i, (clean, variants) in enumerate(res_s4.items()):
        if i >= 3: break
        print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s4)
    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)
    print(f"\n>>> Iteration {iteration + 1} complete. Activities updated.")




>>> ITERATION 1 / 3

>>> STEP 1: Candidate Filtering
>>> Running Step 1  with 10 repetitions...
Candidates found (39 total): ['Check for completeness', 'Deliver card', 'Make decision', 'Notify accept', 'Perform checks'] ...

>>> STEP 2: Context Abstraction
>>> Running Step 2 (Validation Retry Mode, Max: 10)
>>> Step 2 success: All 39 activities summarized.
Sample Context: {'activity': 'Check for completeness', 'predecessors': 'Request / information received', 'successors': 'Additional information request or detailed checks'}

>>> STEP 3: Clustering
>>> Running Step 3  with 10 repetitions...
    Repetition 10/10
>>> Clusters Preview (Total: 10 groups)
  - ['Check for completeness', 'confirm completeness', 'validate completeness', '...']
  - ['Deliver card', 'dispatch card', 'issue card', '...']
  - ['Make decision', 'determine outcome', 'reach decision', '...']
  ... and 7 more groups

>>> STEP 4: Final Mapping
>>> Running Step 4  ...

>>> Mapping Preview (Total: 10 groups)
  - Check f

In [2]:
LOG_NAME = "pub_seed52_ratio0.3_synonymous_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10
fix_repetition = 3
current_df = df.copy()

for iteration in range(fix_repetition):
    print(f"\n{'='*40}")
    print(f">>> ITERATION {iteration + 1} / {fix_repetition}")
    print(f"{'='*40}")

    print("\n>>> STEP 1: Candidate Filtering")
    activity_list_json = json.dumps(current_df['activity'].unique().tolist(), indent=4, ensure_ascii=False)
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, prompts.SYSTEM_PROMPT_SYNONYMOUS_STEP1, prompts.USER_PROMPT_SYNONYMOUS_STEP1)
    
    if not res_s1['data']:
        print(f">>> No more synonymous candidates found in iteration {iteration + 1}. Stopping loop.")
        break
    print(f"Candidates found ({len(res_s1['data'])} total): {res_s1['data'][:5]} ...")

    print("\n>>> STEP 2: Context Abstraction")
    context_json = step2.get_synonym_context(current_df, set(res_s1['data']))
    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, context_json, prompts.SYSTEM_PROMPT_SYNONYM_STEP2, prompts.USER_PROMPT_SYNONYM_STEP2)
    s2_output_json = json.dumps(res_s2["summarized_context"], indent=2, ensure_ascii=False)
    print(f"Sample Context: {res_s2['summarized_context'][0] if res_s2['summarized_context'] else 'None'}")

    print("\n>>> STEP 3: Clustering")
    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, s2_output_json, prompts.SYSTEM_PROMPT_SYNONYM_STEP3, prompts.USER_PROMPT_SYNONYM_STEP3)
    
    if not res_s3:
        print(f">>> No clusters formed in iteration {iteration + 1}.")
        continue
    
    print(f"\n>>> Clusters Preview (Total: {len(res_s3)} groups)")
    for group in res_s3[:3]:
        items = group[:3] + ["..."] if len(group) > 3 else group[:3]
        print(f"  - {items}")
    print(f"  ... and {len(res_s3)-3} more groups")

    print("\n>>> STEP 4: Final Mapping")
    activity_counts = current_df['activity'].value_counts().to_dict()
    res_s4 = step4.run_step4(res_s3, activity_counts)
    
    print(f"\n>>> Mapping Preview (Total: {len(res_s4)} groups)")
    for i, (clean, variants) in enumerate(res_s4.items()):
        if i >= 3: break
        print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s4)
    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)
    print(f"\n>>> Iteration {iteration + 1} complete. Activities updated.")




>>> ITERATION 1 / 3

>>> STEP 1: Candidate Filtering
>>> Running Step 1  with 10 repetitions...
Candidates found (40 total): ['Bring drinks', 'Bring food', 'Deliver to customer', 'Prepare main course', 'Prepare starter'] ...

>>> STEP 2: Context Abstraction
>>> Running Step 2 (Validation Retry Mode, Max: 10)
>>> Step 2 success: All 40 activities summarized.
Sample Context: {'activity': 'Bring drinks', 'predecessors': 'Drink and meal preparation & order verification', 'successors': 'Ongoing drink preparation and meal cooking'}

>>> STEP 3: Clustering
>>> Running Step 3  with 10 repetitions...
    Repetition 10/10
>>> Clusters Preview (Total: 10 groups)
  - ['Bring drinks', 'carry drinks', 'deliver drinks', '...']
  - ['Bring food', 'carry food', 'deliver food', '...']
  - ['Deliver to customer', 'bring to customer', 'hand to customer', '...']
  ... and 7 more groups

>>> STEP 4: Final Mapping
>>> Running Step 4  ...

>>> Mapping Preview (Total: 10 groups)
  - Bring drinks: ['carry drin